In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import grid_helpers as G
import big_view  # bi-modal grid-view


big_view.grid_spec = G.make_grid_spec(x0=10, ncol=8)
big_view.draw_grid('blue')
big_view.mcanvas

MultiCanvas(height=100, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_r…

In [3]:
big_view.show_mode(False)

In [4]:
big_view.fg.line_width

1.0

In [8]:
N = 5
cells = [None] * N*N
colors = {'r': 'red',
          'b': 'blue',
          'g': 'green',
          'y': 'yellow',
          'o': 'orange',
          'k': 'black',
          }

color_key = 'k'


def pos2idx(pos):
    return N*pos[1] + pos[0]


def idx2pos(i):
    row, col = divmod(i, N)
    return (col, row)


def add(pos):
    i = pos2idx(pos)
    cells[i] = color_key
    color = colors[color_key]
    G.fill_rect(big_view.bg, pos, big_view.grid_spec, color=color)


def remove(pos):
    i = pos2idx(pos)
    cells[i] = None
    G.clear_rect(big_view.bg, pos, big_view.grid_spec)


def set_color_key(c):
    global color_key
    color_key = c

In [11]:
def get_component(start, component, get_next_idxs, is_terminal=lambda x: False):
    idxs = {start}
    while idxs:
        i = idxs.pop()
        component.add(i)
        for j in get_next_idxs(i):
            if is_terminal(j):
                component.add(j)
            elif j not in component:
                idxs.add(j)


def distance(i, j):
    c0, r0 = idx2pos(i)
    c, r = idx2pos(j)
    return max(abs(c-c0), abs(r-r0))


def get_next_idxs(i):
    idxs = []

    for di in (-N, -1, 1,  N):
        j = i+di
        if (0 <= j < N*N) and distance(i, j) <= 1 and cells[i] == cells[j]:
            idxs.append(j)

    return idxs

In [13]:
comp = set()
get_component(1, comp, get_next_idxs)

In [14]:
comp

{1, 2, 3, 5, 6, 8, 9, 10, 14, 15, 17, 18, 19, 22}

In [10]:
import big_C as C


C.callbacks = ({
    ('mouse_down', True): (add, ()),
    ('mouse_down', False): (remove, ()),
    ('mouse_move', True): (add, ()),
    ('mouse_move', False): (remove, ()),
    ('c', True): (big_view.bg.clear, ()),
    }
    | {(c, True): (set_color_key, (c,)) for c in colors}
    )


big_view.grid_spec = G.make_grid_spec(x0=10, ncol=N)
big_view.reset()
big_view.draw_grid('blue')

C.canvas = big_view.mcanvas
C.view = big_view
C.attach()
C.show()

MultiCanvas(height=100, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_r…

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

(10, 10, 16.0, 16.0, 5, 5)

In [9]:
C.view.grid_spec

(10, 10, 16.0, 16.0, 5, 5)

In [ ]:
N = 5
cells = [[None]*N for _ in range(N)]
colors = {'r': 'red',
          'b': 'blue',
          'g': 'green',
          'y': 'yellow',
          'o': 'orange',
          'k': 'black',
          }


def set_color_key(c):
    global color_key
    color_key = c


def set_cell(pos, value):
    col, row = pos
    cells[row][col] = value


def get_cell(pos):
    col, row = pos
    return cells[row][col]


def add(pos):
    set_cell(pos, color_key)
    G.fill_rect(big_view.bg, pos, big_view.grid_spec, color=colors[color_key])


def remove(pos):
    set_cell(pos, None)
    G.clear_rect(big_view.bg, pos, big_view.grid_spec)


In [ ]:
# with 2-d arrays

In [16]:
N = 5
cells = [[None]*N for _ in range(N)]
colors = {'r': 'red',
          'b': 'blue',
          'g': 'green',
          'y': 'yellow',
          'o': 'orange',
          'k': 'black',
          }


state = {'color_key': 'k'}


def set_color_key(c):
    global color_key
    color_key = c


def set_cell(pos, value):
    col, row = pos
    cells[row][col] = value


def get_cell(pos):
    col, row = pos
    return cells[row][col]


def add(pos):
    set_cell(pos, color_key)
    G.fill_rect(big_view.bg, pos, big_view.grid_spec, color=colors[color_key])


def remove(pos):
    set_cell(pos, None)
    G.clear_rect(big_view.bg, pos, big_view.grid_spec)


In [17]:
def get_component(start, component, get_next, is_terminal=lambda x: False):
    pts = {start}
    while pts:
        pt0 = pts.pop()
        component.add(pt0)
        for pt in get_next(pt0):
            if is_terminal(pt):
                component.add(pt)
            elif pt not in component:
                pts.add(pt)


def is_inside(pt):
    x, y = pt
    return 0 <= x < N and 0 <= y < N


def get_next(pt):
    x, y = pt
    pts = [(x+1, y), (x-1, y), (x, y+1), (x, y-1)]
    return [pt for pt in pts if is_inside(pt)]